# 数据预处理

## 特征的离散化

In [2]:
import numpy as np
np.random.seed(1)
x = np.random.randn(10) #创建10个标准正态
print(x)

[ 1.62434536 -0.61175641 -0.52817175 -1.07296862  0.86540763 -2.3015387
  1.74481176 -0.7612069   0.3190391  -0.24937038]


### 使用Numpy中的digitize

In [5]:
#创建5个边境构成6个区间
bins = np.linspace(min(x),max(x),5)
print('区间边界为:',bins,sep='')

区间边界为:[-2.3015387  -1.28995108 -0.27836347  0.73322415  1.74481176]


In [6]:
#可以用np.digitize()函数计算特征数据x所属区间的代码
bin_num = np.digitize(x,bins=bins)
print('数据所属区间的编号:',bin_num)

数据所属区间的编号: [4 2 2 2 4 1 5 2 3 3]


In [9]:
#设置不同长度的区间
bins2 = np.array([-2,-1,0.5,1])
#重新计算区间编号
bin_num2 = np.digitize(x,bins = bins2)
print('数据所属的区间编号:',bin_num2)

数据所属的区间编号: [4 2 2 1 3 0 4 2 2 2]


### 使用pandas的cut()离散化

In [11]:
import pandas as pd
import numpy as np
np.random.seed(1)
scores = np.random.randint(0,100,30)
scores

array([37, 12, 72,  9, 75,  5, 79, 64, 16,  1, 76, 71,  6, 25, 50, 20, 18,
       84, 11, 28, 29, 14, 50, 68, 87, 87, 94, 96, 86, 13])

In [13]:
bins = [0,60,70,80,90,np.inf]
pd.cut(scores,bins = bins,right=False)

[[0.0, 60.0), [0.0, 60.0), [70.0, 80.0), [0.0, 60.0), [70.0, 80.0), ..., [80.0, 90.0), [90.0, inf), [90.0, inf), [80.0, 90.0), [0.0, 60.0)]
Length: 30
Categories (5, interval[float64, left]): [[0.0, 60.0) < [60.0, 70.0) < [70.0, 80.0) < [80.0, 90.0) < [90.0, inf)]

In [14]:
pd.cut(scores,bins=bins,right=False,labels = ['不及格','及格','中等','良好','优秀'])


['不及格', '不及格', '中等', '不及格', '中等', ..., '良好', '优秀', '优秀', '良好', '不及格']
Length: 30
Categories (5, object): ['不及格' < '及格' < '中等' < '良好' < '优秀']

In [16]:
df_scores = pd.DataFrame(scores,columns=['c1'])
df_scores.head()

,c1
0,37
1,12
2,72
3,9
4,75


In [18]:
#将分类作为一列加入DF
df_scores['成绩分类'] = pd.cut(scores,bins=bins,right=False,labels = ['不及格','及格','中等','良好','优秀'])
df_scores.head()

,c1,成绩分类
0,37,不及格
1,12,不及格
2,72,中等
3,9,不及格
4,75,中等


In [22]:
df_scores['成绩分类'].value_counts()

成绩分类
不及格    17
中等      5
良好      4
及格      2
优秀      2
Name: count, dtype: int64

## 识别和处理异常值

### 识别

In [23]:
import numpy as np
np.random.seed(1)
x = np.random.randn(10)
x

array([ 1.62434536, -0.61175641, -0.52817175, -1.07296862,  0.86540763,
       -2.3015387 ,  1.74481176, -0.7612069 ,  0.3190391 , -0.24937038])

In [24]:
#将第0个值替换为-1000
x[0] = -1000
#将第五个值替换为异常值
x[5] = 400
#将最后一个值替换为异常值
x[-1] = 500
x

array([-1.00000000e+03, -6.11756414e-01, -5.28171752e-01, -1.07296862e+00,
        8.65407629e-01,  4.00000000e+02,  1.74481176e+00, -7.61206901e-01,
        3.19039096e-01,  5.00000000e+02])

In [25]:
q3,q1 = np.percentile(x,[75,25])#获取x的四分之三位数和四分之一位数
iqr = q3 - q1
n = 2
range_lower = q1 - n*iqr
range_lower

-5.221454298238596

In [26]:
range_upper = q3 + n * iqr
range_upper

6.02257074964828

In [28]:
np.where((x<range_lower) | (x>range_upper))

(array([0, 5, 9], dtype=int64),)

### 处理

#### 法1:忽略异常值所在的样本

In [31]:
df = pd.DataFrame(x,columns=['col'])
df.head()

,col
0,-1000.000000
1,-0.611756
2,-0.528172
3,-1.072969
4,0.865408


In [32]:
df_new = df[(range_lower <=df['col']) & (df['col'] < range_upper)]
df_new

,col
1,-0.611756
2,-0.528172
3,-1.072969
4,0.865408
6,1.744812
7,-0.761207
8,0.319039


#### 法2:大于上限的值直接用上限代替,小于下限的值直接用下限代替

In [38]:
#选出小于上限的值,否则用上限代替
s1 = df['col'].where(df['col'] < range_upper,range_upper)
#选出大于下限的值,否则用下限代替
s2 = s1.where(s1> range_lower,range_lower)
print(s2)

0   -5.221454
1   -0.611756
2   -0.528172
3   -1.072969
4    0.865408
5    6.022571
6    1.744812
7   -0.761207
8    0.319039
9    6.022571
Name: col, dtype: float64


In [39]:
#将筛选结果作为新的一列
df['col_new'] = s2
df

,col,col_new
0,-1000.000000,-5.221454
1,-0.611756,-0.611756
2,-0.528172,-0.528172
3,-1.072969,-1.072969
4,0.865408,0.865408
5,400.000000,6.022571
6,1.744812,1.744812
7,-0.761207,-0.761207
8,0.319039,0.319039
9,500.000000,6.022571


## 特征值的Min-Max缩放

In [41]:
#将特征值转换为0-1之间的
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
np.random.seed(1)
a = np.random.randn(5,2)
a

array([[ 1.62434536, -0.61175641],
       [-0.52817175, -1.07296862],
       [ 0.86540763, -2.3015387 ],
       [ 1.74481176, -0.7612069 ],
       [ 0.3190391 , -0.24937038]])

In [48]:
scaler = MinMaxScaler()
scaler.fit(a)
a_saled = scalar.transform(a)
a_saled

array([[0.94700076, 0.8234131 ],
       [0.        , 0.59866925],
       [0.6131058 , 0.        ],
       [1.        , 0.75058745],
       [0.37273075, 1.        ]])

In [50]:
#还原
a_restored = scaler.inverse_transform(a_saled)
a_restored

array([[ 1.62434536, -0.61175641],
       [-0.52817175, -1.07296862],
       [ 0.86540763, -2.3015387 ],
       [ 1.74481176, -0.7612069 ],
       [ 0.3190391 , -0.24937038]])

## 特征值标准化

In [51]:
#标准化是将特征值缩放为均值为0标准差为1的符合正太分布的数据
from sklearn.preprocessing import StandardScaler
np.random.seed(1)
a = np.random.randn(5,2)
a

array([[ 1.62434536, -0.61175641],
       [-0.52817175, -1.07296862],
       [ 0.86540763, -2.3015387 ],
       [ 1.74481176, -0.7612069 ],
       [ 0.3190391 , -0.24937038]])

In [52]:
scaler = StandardScaler()
scaler.fit(a)

StandardScaler()

In [53]:
a_standard = scaler.transform(a)
a_standard

array([[ 0.96931976,  0.55142609],
       [-1.57746645, -0.10470577],
       [ 0.07137004, -1.85249987],
       [ 1.11185158,  0.33881414],
       [-0.57507493,  1.06696541]])

In [54]:
#回复数据
a_restored = scaler.inverse_transform(a_standard)
a_restored

array([[ 1.62434536, -0.61175641],
       [-0.52817175, -1.07296862],
       [ 0.86540763, -2.3015387 ],
       [ 1.74481176, -0.7612069 ],
       [ 0.3190391 , -0.24937038]])

## 特征值的稳健缩放

In [55]:
#如果存在异常值将会影响均值,方差,标准差
#RobustScaler
from sklearn.preprocessing import RobustScaler
np.random.seed(1)
a = np.random.randn(5,2)
a

array([[ 1.62434536, -0.61175641],
       [-0.52817175, -1.07296862],
       [ 0.86540763, -2.3015387 ],
       [ 1.74481176, -0.7612069 ],
       [ 0.3190391 , -0.24937038]])

In [59]:
scaler = RobustScaler()
scaler.fit(a)

RobustScaler()

In [62]:
a_robusted = scaler.transform(a)
a_robusted

array([[ 0.58142503,  0.32403845],
       [-1.06762636, -0.67596155],
       [ 0.        , -3.33974636],
       [ 0.67371479,  0.        ],
       [-0.41857497,  1.10976361]])

In [63]:
a_restored = scaler.inverse_transform(a_robusted)
a_restored

array([[ 1.62434536, -0.61175641],
       [-0.52817175, -1.07296862],
       [ 0.86540763, -2.3015387 ],
       [ 1.74481176, -0.7612069 ],
       [ 0.3190391 , -0.24937038]])

## 无序分类数据的热编码

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
#设置DF
pd.set_option('display.unicode.east_asian_width',True)
df = pd.DataFrame({'专业':['计算机','信管','计算机'],
                   '组号':[3,1,2],'得分':[80,85,78]})
df

,专业,组号,得分
0,计算机,3,80
1,信管,1,85
2,计算机,2,78


In [2]:
df2 = df[['专业','组号']]
df2

,专业,组号
0,计算机,3
1,信管,1
2,计算机,2


In [3]:
coder = OneHotEncoder(handle_unknown = 'ignore')
coder.fit(df2)

OneHotEncoder(handle_unknown='ignore')

In [4]:
coder.categories_

[array(['信管', '计算机'], dtype=object), array([1, 2, 3], dtype=int64)]

In [5]:
a = coder.transform(df2).toarray()
a

array([[0., 1., 0., 0., 1.],
       [1., 0., 1., 0., 0.],
       [0., 1., 0., 1., 0.]])

In [6]:
coder.get_feature_names_out()

array(['专业_信管', '专业_计算机', '组号_1', '组号_2', '组号_3'], dtype=object)

In [7]:
coder.get_feature_names_out(['专业','组号'])

array(['专业_信管', '专业_计算机', '组号_1', '组号_2', '组号_3'], dtype=object)

In [8]:
#反编译
coder.inverse_transform(a)

array([['计算机', 3],
       ['信管', 1],
       ['计算机', 2]], dtype=object)

In [9]:
#将one-hot编码添加到DF
df[coder.get_feature_names_out(['专业','组号']).tolist()] = a.tolist()
df

,专业,组号,得分,专业_信管,专业_计算机,组号_1,组号_2,组号_3
0,计算机,3,80,0.0,1.0,0.0,0.0,1.0
1,信管,1,85,1.0,0.0,1.0,0.0,0.0
2,计算机,2,78,0.0,1.0,0.0,1.0,0.0


In [11]:
#删除原始的特征值
df.drop(columns=['专业','组号'])

,得分,专业_信管,专业_计算机,组号_1,组号_2,组号_3
0,80,0.0,1.0,0.0,0.0,1.0
1,85,1.0,0.0,1.0,0.0,0.0
2,78,0.0,1.0,0.0,1.0,0.0


In [12]:
#sklearn.preprocessing中的LabelBinarizer类,panndas中的get_dummies()函数都可以实现该功能.
pd.get_dummies(df)

,组号,得分,专业_信管,专业_计算机,组号_1,组号_2,组号_3,专业_信管,专业_计算机
0,3,80,0.0,1.0,0.0,0.0,1.0,False,True
1,1,85,1.0,0.0,1.0,0.0,0.0,True,False
2,2,78,0.0,1.0,0.0,1.0,0.0,False,True


In [13]:
pd.get_dummies(df,columns=['专业'])

,组号,得分,专业_信管,专业_计算机,组号_1,组号_2,组号_3,专业_信管,专业_计算机
0,3,80,0.0,1.0,0.0,0.0,1.0,False,True
1,1,85,1.0,0.0,1.0,0.0,0.0,True,False
2,2,78,0.0,1.0,0.0,1.0,0.0,False,True


In [14]:
pd.get_dummies(df,columns=['专业','组号'])

,得分,专业_信管,专业_计算机,组号_1,组号_2,组号_3,专业_信管,专业_计算机,组号_1,组号_2,组号_3
0,80,0.0,1.0,0.0,0.0,1.0,False,True,False,False,True
1,85,1.0,0.0,1.0,0.0,0.0,True,False,True,False,False
2,78,0.0,1.0,0.0,1.0,0.0,False,True,False,True,False


In [15]:
#对于多标签分类情况,可以使用sklearn,preprocessing中的MultiLabelBinarizer类实现编码
from sklearn.preprocessing import MultiLabelBinarizer
df = pd.DataFrame({'学号':[3,1,2],
                   '主修专业':['计算机','信管','计算机'],
                   '辅修专业':['统计','计算机','人工智能']})
df

,学号,主修专业,辅修专业
0,3,计算机,统计
1,1,信管,计算机
2,2,计算机,人工智能


In [16]:
a = df[['主修专业','辅修专业']].values
a

array([['计算机', '统计'],
       ['信管', '计算机'],
       ['计算机', '人工智能']], dtype=object)

In [17]:
mlb = MultiLabelBinarizer()
mlb.fit(a)


MultiLabelBinarizer()

In [18]:
mlb.transform(a)

array([[0, 0, 1, 1],
       [0, 1, 0, 1],
       [1, 0, 0, 1]])

In [19]:
mlb.classes_

array(['人工智能', '信管', '统计', '计算机'], dtype=object)

In [20]:
mlb = MultiLabelBinarizer(classes=['计算机','人工智能','信管','统计'])
mlb.fit_transform(a)

array([[1, 0, 0, 1],
       [1, 0, 1, 0],
       [1, 1, 0, 0]])

## 有序分类数据编码

In [21]:
#有些类别存在顺序关系,比如考核成绩
df = pd.DataFrame({'学号':[3,1,2],
                   '成绩':['优秀','中等','及格']})
df

,学号,成绩
0,3,优秀
1,1,中等
2,2,及格


In [22]:
mapper = {'优秀':5,'良好':4,'中等':3,'及格':2,'不及格':1}
s = df['成绩'].replace(mapper)
s

C:\Users\86157\AppData\Local\Temp\ipykernel_26124\3881950698.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = df['成绩'].replace(mapper)


0    5
1    3
2    2
Name: 成绩, dtype: int64

In [23]:
df['成绩编码'] = s
df

,学号,成绩,成绩编码
0,3,优秀,5
1,1,中等,3
2,2,及格,2


In [24]:
df.drop(columns=['成绩'])

,学号,成绩编码
0,3,5
1,1,3
2,2,2


## 正则化

In [25]:
#正则化是将每个样本各个特征值缩放到单位范数.p范数
from sklearn.preprocessing import Normalizer
x =[[1,2,3],[1,3,9],[5,8,3]]
scaler = Normalizer()
scaler.fit(x)

Normalizer()

In [26]:
scaler.transform(x)

array([[0.26726124, 0.53452248, 0.80178373],
       [0.10482848, 0.31448545, 0.94345635],
       [0.50507627, 0.80812204, 0.30304576]])